# Clase 027 — concat, merge, join

**Parte 0** · VanderPlas cap. 3 §§ 3.7-3.8.

> 🎯 Juntar datasets sin generar duplicados ni perder filas. SQL-style joins en pandas.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd

## 1️⃣ `concat` — apilado simple

Alinea por index (`axis=0` apila filas) o por columnas (`axis=1`).

In [ ]:
ene = pd.DataFrame({'tienda': ['A','B'], 'monto': [100, 80]})
feb = pd.DataFrame({'tienda': ['A','B'], 'monto': [110, 90]})
mar = pd.DataFrame({'tienda': ['A','B'], 'monto': [105, 95]})

trim = pd.concat([ene, feb, mar], ignore_index=True)
print(trim)

# Con keys para mantener trazabilidad
trim_keys = pd.concat([ene, feb, mar], keys=['ene', 'feb', 'mar'])
print('\ncon keys (MultiIndex):')
print(trim_keys)

## 2️⃣ Los 4 tipos de join (SQL)

```
A  |  B          INNER     LEFT      RIGHT     OUTER
1  |  10         A∩B       A         B         A∪B
2  |  20         (común)   (todo A)  (todo B)  (todos, NaN donde falta)
3  |  --
--|  30
```

In [ ]:
clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3, 4],
    'nombre'    : ['Ana', 'Bob', 'Cris', 'Dan'],
})
ordenes = pd.DataFrame({
    'orden_id'  : [101, 102, 103, 104, 105],
    'cliente_id': [1, 1, 2, 5, 5],   # 5 no está en clientes; 3 y 4 no tienen orden
    'monto'     : [50, 80, 30, 40, 60],
})

print('clientes:')
print(clientes)
print('\nordenes:')
print(ordenes)

In [ ]:
# INNER join: solo clientes CON órdenes
print('--- INNER ---')
print(pd.merge(clientes, ordenes, on='cliente_id', how='inner'))

# LEFT join: TODOS los clientes, NaN si no tienen orden
print('\n--- LEFT ---')
print(pd.merge(clientes, ordenes, on='cliente_id', how='left'))

# OUTER: todos los clientes Y todas las órdenes
print('\n--- OUTER ---')
print(pd.merge(clientes, ordenes, on='cliente_id', how='outer'))

## 3️⃣ `validate` — atajo anti-bugs

Declara qué relación **esperas** (`1:1`, `1:m`, `m:1`, `m:m`); si los datos no la cumplen, pandas lanza excepción **antes** de generar duplicados.

In [ ]:
# Esperamos que cada cliente tenga muchas órdenes (1:m)
result = pd.merge(clientes, ordenes, on='cliente_id', how='left', validate='one_to_many')
print('OK (1:m válido)')

# Si esperaras 1:1 cuando realmente es 1:m, falla:
try:
    pd.merge(clientes, ordenes, on='cliente_id', validate='one_to_one')
except pd.errors.MergeError as e:
    print(f'\nMergeError correcto: {e}')

## 4️⃣ `indicator=True` — auditoría

Agrega columna `_merge` con `'left_only'`, `'right_only'`, `'both'`. Útil para entender qué pasó:

In [ ]:
audit = pd.merge(clientes, ordenes, on='cliente_id', how='outer', indicator=True)
print(audit)
print('\nResumen:')
print(audit['_merge'].value_counts())

## 5️⃣ `df.join` — atajo por index

Cuando ambos tienen el index alineado a la key del join, `df1.join(df2)` es más corto que `merge`:

In [ ]:
c = clientes.set_index('cliente_id')
o = ordenes.set_index('cliente_id')
print('join por index:')
print(c.join(o, how='left'))

## 6️⃣ `on` con columnas distintas: `left_on`/`right_on`

In [ ]:
tablaA = pd.DataFrame({'id_cliente': [1, 2], 'pais': ['ES', 'CL']})
tablaB = pd.DataFrame({'cliente_id': [1, 2], 'plan': ['pro', 'free']})

print(pd.merge(tablaA, tablaB, left_on='id_cliente', right_on='cliente_id'))

## ✅ Checklist

- [ ] Distingo `concat` (apilar) de `merge` (joinear)
- [ ] Elijo inner/left/right/outer según necesidad
- [ ] Uso `validate` para no generar duplicados ocultos
- [ ] Uso `indicator=True` para auditar el merge
- [ ] Conozco `df.join` como atajo por index

## 📝 Homework

Ver `README.md`. 4 joins con indicator, validate, join por index.

## 📖 Definiciones y características

**`concat`**

Apila DataFrames por filas (`axis=0`, default) o columnas (`axis=1`). Alinea por el otro eje. No requiere key; es apilamiento puro.

**`merge` (SQL-style join)**

Combina dos DataFrames por una **key** común. `how='inner'/'left'/'right'/'outer'/'cross'` controla qué filas se conservan.

**INNER JOIN**

Solo filas presentes en AMBOS lados de la key. Si la key no matchea, se descarta. **Default** de `merge`.

**LEFT JOIN**

Todas las filas del lado izquierdo (`df1`). Si no hay match en el derecho, columnas derechas quedan NaN. Útil para enriquecer datos sin perder ninguno.

**OUTER JOIN**

Todas las filas de ambos lados; NaN donde no hay match. Vista "unión". Útil para auditoría.

**`validate`**

Parámetro de `merge` que valida la cardinalidad esperada: `'one_to_one'`, `'one_to_many'`, `'many_to_one'`, `'many_to_many'`. Si los datos no la cumplen, lanza error → evita duplicados ocultos.

**`indicator=True`**

Añade columna `_merge` con `'left_only'`/`'right_only'`/`'both'`. Útil para auditar qué tipo de match tuvo cada fila.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Tras `merge` tengo más filas que el DataFrame original | Relación uno-a-muchos no esperada. **Fix**: `validate='one_to_one'` (lanza error si no es 1:1) o investiga duplicados con `df[df.duplicated('key')]`. |
| `merge` produce `KeyError` en la key | Tipos distintos: `int` vs `str` aunque el valor sea el mismo. **Fix**: `df['id'].astype(str)` en ambos lados antes del merge. |
| Columnas se renombran con `_x`/`_y` tras merge | Ambos DataFrames tenían cols con el mismo nombre (que no era la key). **Fix**: `merge(..., suffixes=('_left', '_right'))` para nombres explícitos. |
| `pd.concat([df1, df2])` da columnas extra con NaN | Los dos tenían columnas distintas (pandas las une todas, llena con NaN). **Fix**: `concat(..., join='inner')` para conservar solo cols comunes. |
| `concat` ignora mi `ignore_index=True` y queda raro | Si tus DFs tienen index distintos, sin `ignore_index=True` mantiene los originales (puede haber duplicados). Default de `concat` es `ignore_index=False`. |

## ❓ Preguntas frecuentes

**❓ ¿`merge` o `join`?**

**`merge`** es la API rica (por columnas, control total). **`df1.join(df2)`** es atajo cuando ambos tienen index alineado a la key. Mismo motor por dentro.

**❓ ¿Cuándo `concat` vs `merge`?**

**`concat`**: apilas datos con la misma estructura (mes 1, mes 2, mes 3 → año). **`merge`**: combinas datasets diferentes que comparten una key (clientes + órdenes).

**❓ ¿`validate` siempre?**

Sí — cuesta nada y atrapa el bug "silenciosamente generé el doble de filas". Recomendado en todo merge de producción.

**❓ ¿Cómo merge por múltiples columnas?**

`merge(df1, df2, on=['a', 'b'])` o `left_on=['a','b'], right_on=['x','y']`. La key compuesta es lista de strings.

**❓ ¿Merge es lento con datasets grandes?**

Con N=1M ya empieza a notarse. Acelera: setea index a la key antes (`set_index('key').join(...)`) o usa DuckDB (`SELECT ... JOIN`) — frecuentemente más rápido.

## 🔗 Referencias

- VanderPlas cap. 3 §§ 3.7-3.8
- [pandas Merge guide](https://pandas.pydata.org/docs/user_guide/merging.html)

➡️ **Siguiente:** [028 — groupby](../028-pandas-groupby-split-apply-combine/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Concat por filas** (3 meses -> anual).

In [ ]:
import pandas as pd
ene = pd.DataFrame({'producto': ['A', 'B'], 'ventas': [10, 20]})
feb = pd.DataFrame({'producto': ['A', 'B'], 'ventas': [15, 25]})
mar = pd.DataFrame({'producto': ['A', 'B'], 'ventas': [12, 22]})
anual = pd.concat([ene, feb, mar], ignore_index=True)
print(anual)
assert anual.shape == (6, 2) and list(anual.index) == list(range(6))

**Ej. 2 — Inner join** (solo clientes con órdenes).

In [ ]:
clientes = pd.DataFrame({'cliente_id': [1, 2, 3], 'nombre': ['Ana', 'Beto', 'Caro']})
ordenes  = pd.DataFrame({'cliente_id': [1, 1, 2], 'monto': [100, 50, 200]})
inner = clientes.merge(ordenes, on='cliente_id', how='inner')
print(inner)
assert set(inner['cliente_id']) == {1, 2}   # Caro (3) no tiene ordenes

**Ej. 3 — Left join** (conserva clientes sin órdenes).

In [ ]:
left = clientes.merge(ordenes, on='cliente_id', how='left')
print(left)
assert left['monto'].isna().sum() == 1       # Caro -> NaN en monto

**Ej. 4 — `validate='one_to_many'`** detecta duplicación oculta.

In [ ]:
clientes_dup = pd.DataFrame({'cliente_id': [1, 1, 2], 'nombre': ['Ana', 'Ana2', 'Beto']})
ok = False
try:
    clientes_dup.merge(ordenes, on='cliente_id', validate='one_to_many')
except Exception as e:
    ok = True
    print('validate detecto llaves duplicadas en el lado izquierdo:', type(e).__name__)
assert ok, 'validate one_to_many debia fallar con llaves izquierdas duplicadas' 

**Ej. 5 — `indicator=True`** para auditar el origen de cada fila.

In [ ]:
c = pd.DataFrame({'id': [1, 2, 3]}); o = pd.DataFrame({'id': [2, 3, 4]})
m = c.merge(o, on='id', how='outer', indicator=True)
print(m['_merge'].value_counts())
assert set(m['_merge'].unique()) == {'left_only', 'right_only', 'both'}